# 1. Bloom Filter Parameter Computation

`mpmt.bf_param(set_size, fpr_mantissa, fpr_exponent)`
Computes Bloom filter parameters given a target set size and false positive rate.

## Parameters

| Parameter | Type | Range | Description |
|------|------|------|------|
| `set_size` | `int` | `[2^10, 2^25]` | Expected set size |
| `fpr_mantissa` | `float` | `[1.0, 10.0)` | FPR mantissa, e.g. `1.0` in `1.0e-3` |
| `fpr_exponent` | `int` | `[-12, -1]` | FPR exponent, e.g. `-3` in `1.0e-3` |

## Returns

`(bf_size, bf_size_log2up, hf_num, hf_num_log2up)`

| Field | Meaning |
|------|------|
| `bf_size` | Bloom filter bit-array length |
| `bf_size_log2up` | `ceil(log2(bf_size))`, the DPF `ell_in` |
| `hf_num` | Number of hash functions |
| `hf_num_log2up` | `floor(log2(hf_num)) + 1`, the upper bound on the ring width for equality_test |

## Formulas

$$bf\_size = \frac{-n \cdot \ln(fpr)}{(\ln 2)^2}, \qquad hf\_num = \frac{bf\_size}{n} \cdot \ln 2$$

In [1]:
import mpmt

# set_size=1024, fpr=1e-3
bf_size, bf_size_log2up, hf_num, hf_num_log2up = mpmt.bf_param(
    set_size=2**10, fpr_mantissa=1.0, fpr_exponent=-3
)
assert bf_size == 14723
assert bf_size_log2up == 14   # 2^14 = 16384 ≥ 14723
assert hf_num == 10
assert hf_num_log2up == 4     # floor(log2(10)) + 1 = 4

print(f"bf_size={bf_size}, log2up={bf_size_log2up}, "
      f"hf_num={hf_num}, hf_log2up={hf_num_log2up}")

bf_size=14723, log2up=14, hf_num=10, hf_log2up=4


## Reference Table — bf_size

Precomputed over the parameter ranges relevant to the paper's experimental settings, for intuitive reference.

Each cell: `bf_size, bf_size_log2up`

| set\fpr | 1e-1 | 1e-2 | 1e-3 | 1e-4 | 1e-5 | 1e-6 |
|---------|------|------|------|------|------|------|
| 2^10 | 4908,13 | 9815,14 | 14723,14 | 19630,15 | 24538,15 | 29445,15 |
| 2^15 | 157042,18 | 314083,19 | 471125,19 | 628166,20 | 785208,20 | 942250,20 |
| 2^20 | 5025331,23 | 10050662,24 | 15075993,24 | 20101324,25 | 25126655,25 | 30151987,25 |
| 2^25 | 160810595,28 | 321621190,29 | 482431784,29 | 643242379,30 | 804052974,30 | 964863569,30 |

| set\fpr | 1e-7 | 1e-8 | 1e-9 | 1e-10 | 1e-11 | 1e-12 |
|---------|------|------|------|------|------|------|
| 2^10 | 34353,16 | 39260,16 | 44168,16 | 49075,16 | 53983,16 | 58891,16 |
| 2^15 | 1099291,21 | 1256333,21 | 1413374,21 | 1570416,21 | 1727458,21 | 1884499,21 |
| 2^20 | 35177318,26 | 40202649,26 | 45227980,26 | 50253311,26 | 55278642,26 | 60303973,26 |
| 2^25 | 1125674163,31 | 1286484758,31 | 1447295353,31 | 1608105948,31 | 1768916542,31 | 1929727137,31 |

## Reference Table — hf_num

`hf_num` depends only on `fpr`, not on `set_size`.

| fpr | 1e-1 | 1e-2 | 1e-3 | 1e-4 | 1e-5 | 1e-6 | 1e-7 | 1e-8 | 1e-9 | 1e-10 | 1e-11 | 1e-12 |
|-----|------|------|------|------|------|------|------|------|------|------|------|------|
| hf_num | 3 | 7 | 10 | 13 | 17 | 20 | 23 | 27 | 30 | 33 | 37 | 40 |
| log2up | 2 | 3 | 4 | 4 | 5 | 5 | 5 | 5 | 5 | 6 | 6 | 6 |

# 2. Cryptographic Primitives

The project assumes a 128-bit computational security parameter by default.

## Random Key

| Function | Semantics |
|------|------|
| `get_key_128bits()` | Cryptographically secure random 128-bit (16-byte) key; random tape based on libsodium |

Returns `bytes` of length 16. Used in the Bloom filter protocol to generate hash seeds.

In [ ]:
import mpmt

key = mpmt.get_key_128bits()
assert len(key) == 16
assert isinstance(key, bytes)

k1 = mpmt.get_key_128bits()
k2 = mpmt.get_key_128bits()
assert k1 != k2

print(f"key: {key.hex()}")

key: 6634135a8f9a7681d7aff32be8f944b1


## AES-DM Hash

`hash_aes_dm(preimage, key, ell)` — Local hash based on AES-DM (Davies-Meyer).

| Parameter | Type | Description |
|------|------|------|
| `preimage` | `bytes \| str` | Preimage; str is encoded as UTF-8 |
| `key` | `bytes \| str` | 128-bit key; zero-padded internally to 16 bytes |
| `ell` | `int` | Output ring bit-width |

Returns a value in `[0, 2^ell)`.

In Bloom filter construction, each hash function is parameterized by an independent seed, truncated to `Z_{2^ell}`, and reduced modulo the BF size to obtain the index.

In [ ]:
import mpmt

ell = 20

seed = mpmt.get_key_128bits()

# bytes preimage
h1 = mpmt.hash_aes_dm(preimage=b"alice", key=seed, ell=ell)
assert 0 <= h1 < (1 << ell)
print(f"0 <= {h1} < 2^{ell}")

# str preimage (UTF-8 encoded)
# Different preimage, same seed
h2 = mpmt.hash_aes_dm(preimage="bob", key=seed, ell=ell)
assert 0 <= h2 < (1 << ell)
print(f"0 <= {h2} < 2^{ell}")
assert h1 != h2
print(f"{h1} != {h2}")

# Same preimage, different seed
seed2 = mpmt.get_key_128bits()
h3 = mpmt.hash_aes_dm(preimage=b"alice", key=seed2, ell=ell)
assert h1 != h3
print(f"{h1} != {h3}")